In [34]:
# Install required packages
!pip install pandas numpy matplotlib seaborn scikit-learn plotly openpyxl scipy statsmodels nbformat


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
# --- RELATIVE PATH SETUP ---
# 1. Get the directory where this script is currently located
#    (Uses Path.cwd() as a fallback if running in Jupyter/Interactive mode)
script_dir = Path(__file__).parent if '__file__' in locals() else Path.cwd()
# 2. Define the data folder relative to the script
#    Ensure your data folder is inside the same directory as this script
relative_data_path = "datas_for_research_vicious_circle_project-20260114T085009Z-3-001/datas_for_research_vicious_circle_project"
BASE_PATH = script_dir / relative_data_path
# 3. Validation (Optional but recommended)
if BASE_PATH.exists():
    print(f"Success! Data directory found at: {BASE_PATH}")
    os.chdir(BASE_PATH)
else:
    print(f"Error: Could not find data directory at: {BASE_PATH}")
    print("Please ensure the 'datas_for_research...' folder is in the same directory as this script.")
# ---------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import plotly.express as px
import plotly.graph_objects as go






Error: Could not find data directory at: c:\Users\ofeki\Desktop\TovTech\rvc1\rvc\datas_for_research_vicious_circle_project-20260114T085009Z-3-001\datas_for_research_vicious_circle_project\datas_for_research_vicious_circle_project-20260114T085009Z-3-001\datas_for_research_vicious_circle_project
Please ensure the 'datas_for_research...' folder is in the same directory as this script.


**Data Loading:**

A function that centralizes the loading of all raw source files (Excel/CSV) into a single dictionary for easy access and management.


In [36]:
def load_data(paths):
    return {
        "benefits": pd.read_excel(paths["benefits"], header = None),
        "lamas": pd.read_excel(paths["lamas"], sheet_name="נתונים פיזיים ונתוני אוכלוסייה ", header = None),
        "socio_regional": pd.read_excel(paths["socio_regional"], header = None),
        "periph_regional": pd.read_excel(paths["periph_regional"], header = None),
        "coordinates": pd.read_csv(paths["coordinates"])
    }

**Merge CBS Data (Base Layer):**

Merges the main benefits table with Central Bureau of Statistics (CBS) data (social and peripheral indices), while strictly validating that no rows are lost during the process.


In [37]:
def merge_lamas(df_benefits, df_lamas):
    before = len(df_benefits)

    df = df_benefits.merge(
        df_lamas[[
            "settlement_symbol",
            "socio_economic_index_cluster",
            "socio_economic_index_score",
            "peripherality_index_cluster",
            "peripherality_index_score"
        ]],
        on="settlement_symbol",
        how="left"
    )
    # Safety check: Ensure row count remains unchanged
    assert len(df) == before, (
        "❌ Lost rows in LAMAS merge"
        f"before={before}, after={len(df)}"
    )

    return df

**Regional Council Data Enrichment:**

A function to fill missing index values (e.g., socio-economic scores) using dedicated regional council datasets, acting as a fallback mechanism without overwriting existing data.

In [38]:
def merge_index_from_regional(df_main, df_regional, index_cols, key="settlement_symbol"):
    """
    Completes missing index values in df_main using df_regional,
    without overwriting existing data. Ensures no rows are lost or added during merge.
    """
    before = len(df_main)


    df = df_main.merge(
        df_regional[[key] + index_cols],
        on=key,
        how="left",
        suffixes=("", "_reg")
    )
    # Safety check: Ensure row count remains unchanged
    assert len(df) == before, (
        f"❌ Row count mismatch after merge on '{key}': "
        f"before={before}, after={len(df)}"
    )
    # Fill missing values in main columns with data from regional columns
    for col in index_cols:
        df[col] = df[col].combine_first(df[f"{col}_reg"])

    # Cleaning auxiliary columns
    df = df.drop(columns=[f"{col}_reg" for col in index_cols])

    return df

**Geospatial Enrichment**:

Enriches the dataset with geographic coordinates (latitude/longitude) to enable mapping and spatial analysis.


In [39]:
def merge_coordinates(df_main, df_coordinates):
    before = len(df_main)
    df = df_main.merge(
        df_coordinates[[
            'settlement_code',
            'lat',
            'lon'
        ]],
        left_on="settlement_symbol",
        right_on="settlement_code",
        how="left"
    )
    df = df.drop(columns=["settlement_code"])

    assert len(df) == before, (
        "❌ Lost rows in COORDINATES merge"
        f"before={before}, after={len(df)}"
    )
    return df

**Data Cleaning & Type Conversion**:

The core cleaning function: removes irrelevant columns, casts data types, and handles "censored" values (asterisks) using statistical imputation.


In [40]:
def clean_values(df):
    # Drop rows where critical socio-economic and peripherality data is entirely missing
    socio_economic_peripherality_cols = df.columns[-6:-2]
    df = df.dropna(subset=socio_economic_peripherality_cols, how="all")

    # Removing unnecessary columns
    cols_to_drop = [
        "settlement_type",
        "injury_allowance",
        "recipients_of_the_senior_citizen_pension_only",
        "recipients_of_the_pension_with_income_supplementation",
        "total_recipients_of_old_age_and/or_survivors’_benefits",
        "num_families_receiving_child_benefit",
        "num_children_receiving_child_benefit",
        "families_with_4+_children_receiving_child_benefit",
        "maternity_benefits",
        "alimony"
    ]

    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    # Define column types
    categorial_cols = ["socio_economic_index_cluster","peripherality_index_cluster"]
    numeric_cols = df.loc[:, 'total_population':'unemployment_benefit'].columns
    float_cols = ["socio_economic_index_score","peripherality_index_score", "lon", "lat"]

    for col in df.columns:
        if col in categorial_cols:
          df[col] = df[col].astype("category")
        elif col in numeric_cols:
          # Clean data: Replace censored values (***, ..) with 5 (median of 0-10 range) and remove commas
          df[col] = df[col].astype(str).replace({r'\*\*\*': '5', r'\.\.': '5'}, regex=True).str.replace(",", "", regex=False)
          df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        elif col in float_cols:
          df[col] = df[col].astype(float)
        else:
          df[col] = df[col].astype("object")
    return df

**Export Data**:

Saves the processed dataset in CSV format for readability and Pickle format to preserve exact data types.


In [41]:
def save_dataset(df, name, output_dir="data/processed"):
    """
    Save DataFrame both as CSV and Pickle.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / f"{name}.csv"
    pkl_path = output_dir / f"{name}.pkl"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_pickle(pkl_path)

    print(f"✅ Saved CSV: {csv_path}")
    print(f"✅ Saved Pickle: {pkl_path}")


**Initialization & Pre-processing**:

Defines file paths, loads raw data, and strips structural metadata (unnecessary header rows) from source files to prepare for merging.


In [42]:
paths = {
    "benefits": "benefits_2024_12.xlsx",
    "lamas": "p_libud_23.xlsx",
    "socio_regional": "24_24_230t3.xlsx",
    "periph_regional": "24_22_420t3.xlsx",
    "coordinates": "israel_settlements_all_with_coords.csv"
}

dfs = load_data(paths)

# Slice rows to skip metadata headers provided by source agencies
df_benefits = dfs["benefits"].iloc[5:].copy()
df_benefits = df_benefits.reset_index(drop=True)
df_benefits.columns = [
    "settlement_name",
    "settlement_symbol",
    "settlement_type",
    "total_population",
    "population_0_17",
    "population_18_64",
    "population_65_plus",
    "total_recipients_of_old_age_and/or_survivors’_benefits",
    "recipients_of_the_pension_with_income_supplementation",
    "recipients_of_the_senior_citizen_pension_only",
    "long_term_care_benefit",
    "general_disability_benefit",
    "special_services_for_persons_with_disabilities",
    "disabled_child_benefit",
    "mobility_benefit",
    "work_injury_victims_receiving_disability_and_dependents’_benefits",
    "injury_allowance",
    "num_families_receiving_child_benefit",
    "num_children_receiving_child_benefit",
    "families_with_4+_children_receiving_child_benefit",
    "maternity_benefits",
    "alimony",
    "income_support_benefit",
    "unemployment_benefit"
]

df_lamas = dfs["lamas"].iloc[9:].copy()
df_lamas = df_lamas.reset_index(drop=True)
df_lamas = df_lamas[df_lamas[3] != 'מועצה אזורית']
df_lamas.rename(columns={1: "settlement_symbol", 250: "socio_economic_index_cluster",
                         251: "socio_economic_index_score",256: "peripherality_index_cluster",
                         257: "peripherality_index_score"}, inplace=True)

df_socio = dfs["socio_regional"].iloc[10:].copy()
df_socio = df_socio.reset_index(drop=True)
df_socio = df_socio.iloc[:-8]
df_socio.rename(columns={5: "settlement_symbol", 12: "socio_economic_index_cluster", 10: "socio_economic_index_score"}, inplace=True)

df_periph = dfs["periph_regional"].iloc[9:].copy()
df_periph = df_periph.reset_index(drop=True)
df_periph = df_periph.iloc[:-4]
df_periph.rename(columns={4: "settlement_symbol", 12: "peripherality_index_cluster", 10: "peripherality_index_score"}, inplace=True)

**Pipeline Execution**:

Runs the full ETL sequence (merging, enrichment, cleaning) and saves the final research-ready dataset.


In [43]:
# Execute Merge Pipeline
data_master = merge_lamas(df_benefits, df_lamas)
data_master = merge_index_from_regional(data_master, df_socio,
    index_cols=[
        "socio_economic_index_cluster",
        "socio_economic_index_score"
    ]
)
data_master = merge_index_from_regional(data_master, df_periph,
    index_cols=[
        "peripherality_index_cluster",
        "peripherality_index_score"
    ]
)
data_master = merge_coordinates(data_master, dfs["coordinates"])
data_master = clean_values(data_master)
save_dataset(data_master, "benefits_final")

✅ Saved CSV: data\processed\benefits_final.csv
✅ Saved Pickle: data\processed\benefits_final.pkl


In [44]:
data_master

,settlement_name,settlement_symbol,total_population,population_0_17,population_18_64,population_65_plus,long_term_care_benefit,general_disability_benefit,special_services_for_persons_with_disabilities,disabled_child_benefit,mobility_benefit,work_injury_victims_receiving_disability_and_dependents’_benefits,income_support_benefit,unemployment_benefit,socio_economic_index_cluster,socio_economic_index_score,peripherality_index_cluster,peripherality_index_score,lat,lon
0,אבו גוש,472,8426,2764,5014,648,296,403,125,126,40,78,86,54,3,-0.711088,6,0.810287,31.805997,35.110069
1,אבו סנאן,473,14502,4106,9235,1161,381,722,236,121,126,136,184,148,3,-0.654802,4,-0.211695,32.960995,35.169399
2,אבו תלול,1375,2867,1653,1175,39,20,69,14,61,11,5,79,15,1,-3.012259,2,-0.774045,31.184164,34.930459
3,אבטין,652,2931,981,1778,172,65,171,56,66,17,37,26,22,2,-1.363299,4,0.006656,32.761513,35.113343
4,אבן יהודה,182,14088,4319,7593,2176,422,262,73,144,71,82,5,118,9,1.629981,7,1.215616,32.271051,34.885743
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274,תל אביב -יפו,5000,461649,98237,286889,76523,17463,15156,3050,4472,1893,2774,1368,5151,8,1.292524,10,4.190096,32.076349,34.789403
275,תל מונד,154,14760,4669,8555,1536,252,323,75,193,78,94,16,122,9,1.472223,6,0.997348,32.252731,34.917324
277,תל שבע,1054,24242,11846,11867,529,329,897,275,462,139,78,679,112,1,-2.409863,3,-0.440129,31.248483,34.860568
278,תפרח,709,2277,1153,977,147,30,78,5,62,5,5,5,5,1,-2.222170,3,-0.441916,31.325338,34.678109


In [45]:
data_master['general_disability_rate'] = round((data_master['general_disability_benefit']/data_master['population_18_64']) * 100,2)
data_master['special_services_disability_rate']= round((data_master['special_services_for_persons_with_disabilities']/data_master['population_18_64']) * 100,2)
data_master['mobility_disability_rate'] = round((data_master['mobility_benefit']/data_master['population_18_64']) * 100,2)
data_master['income_support_rate'] = round((data_master['income_support_benefit']/data_master['population_18_64']) * 100,2)
data_master['long_term_care_rate'] = round((data_master['long_term_care_benefit']/data_master['population_65_plus']) * 100,2)
data_master['unemployment_rate'] = round((data_master['unemployment_benefit']/data_master['population_18_64']) * 100,2)
data_master['work_injury_victims_rate'] = round((data_master['work_injury_victims_receiving_disability_and_dependents’_benefits']/data_master['population_18_64']) * 100,2)
data_master['disabled_child_benefit_rate'] = round((data_master['disabled_child_benefit']/data_master['population_0_17']) * 100,2)

In [46]:
data_master.dtypes


settlement_name                                                        object
settlement_symbol                                                      object
total_population                                                        Int64
population_0_17                                                         Int64
population_18_64                                                        Int64
population_65_plus                                                      Int64
long_term_care_benefit                                                  Int64
general_disability_benefit                                              Int64
special_services_for_persons_with_disabilities                          Int64
disabled_child_benefit                                                  Int64
mobility_benefit                                                        Int64
work_injury_victims_receiving_disability_and_dependents’_benefits       Int64
income_support_benefit                                          

In [47]:
# @title Socio-Economic Cluster vs. General Disability Rate vs. Unemployment Rate in each settlement (over 2000 pop)
# 1. הכנת הדאטה (זהה למקור)
cols = ["lat", "lon", "settlement_name", "total_population",
        "socio_economic_index_cluster", "general_disability_rate", "income_support_rate"]
df = data_master.dropna(subset=cols).copy()

# המרות טיפוסים
df["cluster"] = df["socio_economic_index_cluster"].astype(int)
df["disability"] = df["general_disability_rate"].astype(float)
df["income_support"] = df["income_support_rate"].astype(float)
df["population"] = df["total_population"].astype(int)

# 2. הגדרות צבעים וסקאלות
# --- סקאלה דיסקרטית לקלאסטרים ---
colors = px.colors.sample_colorscale(
    [[0.0, "#d73027"], [0.5, "#c9DD00"], [1.0, "#1a9850"]],
    np.linspace(0, 1, 10)
)
cluster_scale = [[i/9, c] for i, c in enumerate(colors) for _ in range(2)] # הכפלה ליצירת מדרגות

# --- סקאלות רציפות ---
scale_disability = [[0.0, "#1a9850"], [0.45, "#fee08b"], [0.75, "#d73027"], [1.0, "#000000"]]
scale_income_support = "Viridis"

# 3. חישובים גלובליים (SizeRef & CustomData)
vmax = df["population"].max()
sizeref = 2.0 * vmax / (38 ** 2) if vmax > 0 else 1
# סדר: [cluster, disability, unemployment, population]
customdata = df[["cluster", "disability", "income_support", "population"]].to_numpy()

# 4. הגדרת התצוגות (Configuration Dictionary)
# כאן אנחנו מגדירים את ההבדלים בין המצבים במקום לשכפל קוד
views = [
    {
        "label": "Socio-Economic Cluster",
        "color_data": ((df["cluster"] - 1) / 9).to_numpy(), # נרמול ל-0-1 עבור הסקאלה הדיסקרטית
        "colorscale": cluster_scale,
        "cmin": 0, "cmax": 1,
        "title": "Cluster (1–10)",
        "tickmode": "array",
        "tickvals": [i / 9 for i in range(10)],
        "ticktext": [str(i) for i in range(1, 11)],
        "hover_fmt": "%{customdata[0]}", # אינדקס 0 ב-customdata
    },
    {
        "label": "General Disability Rate",
        "color_data": df["disability"].to_numpy(),
        "colorscale": scale_disability,
        "cmin": df["disability"].min(), "cmax": df["disability"].max(),
        "title": "General Disability Rate (%)",
        "tickmode": "auto", "tickvals": None, "ticktext": None,
        "hover_fmt": "General Disability Rate: %{customdata[1]:.2f}%", # אינדקס 1
    },
    {
        "label": "Income Support Rate",
        "color_data": df["income_support"].to_numpy(),
        "colorscale": scale_income_support,
        "cmin": df["income_support"].min(), "cmax": 4,
        "title": "Income Support Rate (%)",
        "tickmode": "auto", "tickvals": None, "ticktext": None,
        "hover_fmt": "Income Support Rate: %{customdata[2]:.2f}%", # אינדקס 2
    }
]

# 5. יצירת הגרף ההתחלתי (לפי התצוגה הראשונה - Cluster)
init_view = views[0]
fig = go.Figure(go.Scattermap(
    lat=df["lat"], lon=df["lon"], mode="markers", text=df["settlement_name"],
    customdata=customdata,
    hovertemplate=f"<b>%{{text}}</b><br>{init_view['label']}: {init_view['hover_fmt']}<br>Population: %{{customdata[3]:,}}<extra></extra>",
    marker=dict(
        size=df["population"], sizemode="area", sizeref=sizeref, opacity=0.9,
        color=init_view["color_data"],
        colorscale=init_view["colorscale"],
        cmin=init_view["cmin"], cmax=init_view["cmax"],
        showscale=True,
        colorbar=dict(
            title=init_view["title"], tickmode=init_view["tickmode"],
            tickvals=init_view["tickvals"], ticktext=init_view["ticktext"],
            len=0.85, x=1.02, xanchor="left"
        )
    )
))

# 6. בניית הכפתורים בלולאה
buttons = []
for view in views:
    buttons.append(dict(
        label=view["label"],
        method="restyle",
        args=[{
            "marker.color": [view["color_data"]],
            "marker.colorscale": [view["colorscale"]],
            "marker.cmin": [view["cmin"]],
            "marker.cmax": [view["cmax"]],
            "marker.colorbar.title": [view["title"]],
            "marker.colorbar.tickmode": [view["tickmode"]],
            "marker.colorbar.tickvals": [view["tickvals"]],
            "marker.colorbar.ticktext": [view["ticktext"]],
            "hovertemplate": [f"<b>%{{text}}</b><br>{view['label']}: {view['hover_fmt']}<br>Population: %{{customdata[3]:,}}<extra></extra>"]
        }]
    ))

# 7. הגדרות Layout סופיות
fig.update_layout(
    map=dict(style="carto-positron", center=dict(lat=31.3, lon=34.9), zoom=6.8),
    width=500, height=900, margin=dict(l=0, r=140, t=20, b=0),
    updatemenus=[dict(
        type="dropdown", direction="down", x=0.0, y=1.0, xanchor="left", yanchor="top",
        showactive=True, font=dict(size=14), pad=dict(t=10, b=10, r=10, l=10),
        buttons=buttons
    )]
)

fig.show()


🗺️ **Interactive Map: Geography, Economy & Health**

**Visualization Logic:**

*   Dropdown Menu: Switches the color layer between Socio-Economic Cluster, Disability Rate, and Income Support Rate.
*   Bubble Size: Represents the population size of each locality.


*   Color Scale:

Cluster: Discrete (Red=Low, Green=High).

Rates: Continuous (Green=Low, Red/Black=High).

**Analytical Goal:**
To visualize spatial patterns and allow immediate comparison between a locality's economic status and its health/income support outcomes without changing the map structure.

We can see that in the periphery (north and south) there are higher rates of settlements with low socioeconomic status (clusters 1-4), while in the center (Tel Aviv District and Central District) there is a high concentration of large settlements with high socioeconomic status (clusters 7-10).

In the center, the rate of recipients of general disability benefits (among the working-age population) is generally lower than the rate of recipients of general disability benefits in cities in the periphery. We see a trend that the further away from Tel Aviv District the rate of recipients of general disability benefits in a locality increases.

Regarding income support benefits, there is a trend (but not a strong trend) that the further away from the center the income support rates in a settlement increase. There are Bedouin settlements in the northern Negev where there are particularly exceptional values ​​of recipients of income support.



In [48]:
social_cols = [
    'socio_economic_index_score',
    'peripherality_index_score',
    'income_support_rate'
]

df_social = data_master[social_cols].copy()

# היפוך כיוון למדדים "חיוביים"
#df_social['socio_economic_index_score'] *= -1
#df_social['peripherality_index_score'] *= -1
df_social['income_support_rate'] *= -1


data_master['social_index'] = StandardScaler().fit_transform(df_social).mean(axis=1)


In [49]:
health_cols = [
    'general_disability_rate',
    'special_services_disability_rate',
    'mobility_disability_rate'
]

df_health = data_master[health_cols].copy()
df_health['general_disability_rate'] *= -1
df_health['special_services_disability_rate'] *= -1
df_health['mobility_disability_rate'] *= -1
data_master['health_index'] = StandardScaler().fit_transform(df_health).mean(axis=1)

In [50]:
# @title Social Index vs. Health Index in each settlement
df = data_master.copy()

cols = [
    'social_index',
    'health_index',
    'socio_economic_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=cols)

fig = px.scatter(
    df,
    x='social_index',
    y='health_index',
    hover_name='settlement_name',
   # size='total_population',
   # size_max = 40,
    color='socio_economic_index_cluster',
    color_continuous_scale='Viridis',
    trendline='ols',
    labels={
        'social_index': 'Social-Economic Vulnerability Index',
        'health_index': 'Disability Health Index',
        'total_population': 'Population size',
        'socio_economic_index_cluster': 'Socio-Economic Index Cluster'
    },
    title='Correlation Between Social-Economic Vulnerability and Disability'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),   # או 'black'
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between the Social Index and the Health Index is {df['social_index'].corr(df['health_index'], method= 'spearman'):.2f}")

The correlation between the Social Index and the Health Index is 0.56


In [33]:
# @title Social Index vs. Health Index - Custom Colors (Green=Resilient, Red=Distress)
import plotly.express as px
import pandas as pd
from sklearn.linear_model import LinearRegression

df = data_master.copy()

cols = ['social_index', 'health_index']
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=cols)

# 1. רגרסיה
X = df[['social_index']]
y = df['health_index']
model = LinearRegression().fit(X, y)

# 2. חישוב שאריות
df['predicted_health'] = model.predict(X)
df['residual'] = df['health_index'] - df['predicted_health']

# --- הגדרת צבעים מותאמת אישית ---
# 0.0 = הערך הכי נמוך (למטה) -> ירוק (חסינות)
# 0.5 = האמצע -> צהוב זהב (לא בהיר מדי)
# 1.0 = הערך הכי גבוה (למעלה) -> אדום (מצוקה)
custom_colors = [
    [0.0, "#cd2626"],  # ירוק עמוק (Resilient)
    [0.5, "#FFC300"],  # צהוב-זהב (Normal)
    [1.0, "#1a9850"]   # אדום (Distress)
]

# 3. יצירת הגרף
fig = px.scatter(
    df,
    x='social_index',
    y='health_index',
    hover_name='settlement_name',
    hover_data={
        'residual': ':.2f',
        'socio_economic_index_cluster': True,
        'social_index': False,
        'health_index': False
    },
   # size='total_population', 
   # size_max=40,

    # --- צביעה ---
    color='residual',
    color_continuous_scale=custom_colors, # השימוש בסקאלה שיצרנו

    trendline='ols',
    trendline_color_override='black',

    title='Resilience vs. Distress: Deviation from Expected Health Index',
    labels={
        'social_index': 'Social Vulnerability (Low = Poor/Peripheral)',
        'health_index': 'Disability Severity (Low = More Disabled)',
        'residual': 'Deviation (Severity)'
    }
)

# 4. עיצוב המקרא (Legend)
fig.update_layout(
    title_font_size=20,
    coloraxis_colorbar=dict(
        title="Anomaly Type",
        # מגדירים את הטקסט שיופיע ליד הצבעים
        tickvals=[-1, 0, 1],
        ticktext=[
            "Distress\n(Worse than expected)", 
            "Normal", 
            "Resilient \n(Better than expected)"
        ]
    )
)

fig.show()

# 5. הדפסת התובנות (מתוקן לוגית)
print('Settlements that "break the rules":')

print("\n--- Top 5 'Resilient' Cities (Green/Positive Residual) ---")
print("Cities with POOR social status but GOOD health outcomes (Low Disability):")
# מיון עולה: הערכים הכי שליליים (הכי למטה בגרף) הם החסינים ביותר
print(df.sort_values('residual', ascending=False).head(10)[['settlement_name', 'socio_economic_index_cluster', 'residual', 'total_population']])

print("\n--- Top 5 'Distress' Cities (Red/Negative Residual) ---")
print("Cities with BETTER social status but BAD health outcomes (High Disability):")
# מיון יורד: הערכים הכי חיוביים (הכי למעלה בגרף) הם במצוקה הגדולה ביותר
print(df.sort_values('residual').head(10)[['settlement_name', 'socio_economic_index_cluster', 'residual', 'total_population']])

Settlements that "break the rules":

--- Top 5 'Resilient' Cities (Green/Positive Residual) ---
Cities with POOR social status but GOOD health outcomes (Low Disability):
    settlement_name socio_economic_index_cluster  residual  total_population
233        קצר א-סר                            1  2.234405              2704
103           טלמון                            5  1.856769              5886
209            עפרה                            6  1.627476              3210
54           ברוכין                            6  1.569153              2637
119       כוכב השחר                            4  1.562317              2850
121       כוכב יעקב                            2  1.545247              3485
2          אבו תלול                            1  1.533905              2867
205         עלי זהב                            7  1.520309              5354
112            יקיר                            7  1.500515              2793
38        ביר הדאג'                            1  1.491831  

### 🎯 Executive Summary: Key Anomalies

*   **⚠️ The "False Positive" (Bedouin Sector):**
    Cluster 1 localities (e.g., *Qasr al-Sir, Abu Talul*) show deceptively low disability rates. This indicates **barriers to access** (under-utilization of rights), not better health.
    > **Action:** Do not cut funding. Instead, launch "Rights Exhaustion" campaigns (מיצוי זכויות) in these specific areas.

*   **🧲 The "Service Magnet" Effect (Regional Hubs):**
    Cities like *Tiberias* and *Be'er Sheva* suffer from "excess disability" (Red Zone). They act as regional hubs, attracting vulnerable populations seeking medical services and public housing.
    > **Action:** These cities carry a "regional burden." They require additional government compensation beyond standard per-capita allocation to support this imported welfare load.

*   **🛡️ Community Resilience:**
    Settlements like *Talmon* and *Ofra* significantly outperform the model. Strong community support networks and younger demographics likely prevent the slide into welfare dependency.

In [51]:
# @title General Disability Rate vs. Income Support Rate in each settlement
df = data_master.copy()

cols = [
    'general_disability_rate',
    'income_support_rate',
    'socio_economic_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df[cols] = df[cols].astype(float)
df = df.dropna(subset= cols)

fig = px.scatter(
    df,
    x='general_disability_rate',
    y='income_support_rate',
    hover_name='settlement_name',
  #  size='total_population',
  #  size_max= 50,
    color='socio_economic_index_cluster',
    color_continuous_scale='Viridis',
    trendline='ols',
    labels={
        'general_disability_rate': 'General Disability Rate (%)',
        'income_support_rate': 'Income Support Rate (%)',
        'total_population': 'Population size',
        'socio_economic_index_cluster': 'Socio-Economic Index Cluster'
    },
    title='Correlation Between General Disability Rate and Income Support Rate'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between general disability rate and income support rate  is {data_master['general_disability_rate'].corr(data_master['income_support_rate'], method= 'spearman'):.2f}")

The correlation between general disability rate and income support rate  is 0.72


In [52]:
# @title General Disability Rate vs. Socio-Economic Index in each settlement

df = data_master.copy()

cols = [
    'general_disability_rate',
    'socio_economic_index_score',
    'peripherality_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset= cols)

fig = px.scatter(
    df,
    x='general_disability_rate',
    y='socio_economic_index_score',
    hover_name='settlement_name',
 #   size='total_population',
 #   size_max=40,
    color='peripherality_index_cluster',
    color_continuous_scale='Plasma',
    trendline='ols',
    labels={
        'general_disability_rate': 'General Disability Rate (%)',
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'total_population': 'Population size',
        'peripherality_index_cluster': 'Peripherality Index'
    },
    title='Correlation Between General Disability Rate and Socio-Economic Index'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between general disability rate and socio economic index  is {data_master['general_disability_rate'].corr(data_master['socio_economic_index_score'], method= 'spearman'):.2f}")

The correlation between general disability rate and socio economic index  is -0.54


In [54]:
# @title General Disability Rate vs. Socio-Economic Index - Deviation from Expected Disability Rate (residuals)


df = data_master.copy()

cols = [
    'general_disability_rate',
    'socio_economic_index_score'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=cols)
# 1. הכנת הנתונים לרגרסיה
X = df[['general_disability_rate']]
y = df['socio_economic_index_score']

model = LinearRegression().fit(X, y)

# 2. חישוב השאריות (Residuals)
# השארית היא המרחק בין המציאות לבין המודל
df['predicted_socio_economic_index'] = model.predict(X)
df['residual'] = df['socio_economic_index_score'] - df['predicted_socio_economic_index']

custom_colors = [
    [0.0, "#cd2626"],  # ירוק עמוק (Resilient)
    [0.5, "#FFC300"],  # צהוב-זהב (Normal)
    [1.0, "#1a9850"]   # אדום (Distress)
]
# 3. יצירת הגרף המשודרג
fig = px.scatter(
    df,
    x='general_disability_rate',
    y='socio_economic_index_score',
    hover_name='settlement_name',
    hover_data={
        'residual': ':.2f',
        'socio_economic_index_cluster': True,
        'general_disability_rate': False, # מסתירים כי זה כבר בציר
        'socio_economic_index_score': False  # מסתירים כי זה כבר בציר
    },
 #   size='total_population',
 #   size_max=40,

    # --- צביעה לפי חריגות ---
    color='residual',
    color_continuous_scale=custom_colors,

    # הוספת קו המגמה השחור
    trendline='ols',
    trendline_color_override='black',

    title='Resilience vs. Distress: Deviation from Expected Disability Rate',
    labels={
        'general_disability_rate': 'General Disability Rate (%)',
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'residual': 'Deviation (Severity)'
    }
)

# 4. עיצוב סופי
fig.update_layout(
    title_font_size=20,
    coloraxis_colorbar=dict(
        title="Anomaly Type",
        tickvals=[-4, 0, 4],
        ticktext=["Resilient\n(Better than expected)", "Normal", "Distress\n(Worse than expected)"]
    )
)

fig.show()

print('---Settlements that "break the rules"---')
print("Settlements that manage to maintain below-average disability rates despite BAD Socio-Economic Index:")
df[['settlement_name', 'socio_economic_index_cluster', 'residual', 'total_population']].sort_values(by = 'residual').head(10)



---Settlements that "break the rules"---
Settlements that manage to maintain below-average disability rates despite BAD Socio-Economic Index:


,settlement_name,socio_economic_index_cluster,residual,total_population
38,ביר הדאג',1,-3.558871,3666
194,סעוה,1,-2.970185,2332
2,אבו תלול,1,-2.945115,2867
233,קצר א-סר,1,-2.830902,2704
152,מודיעין עילית,1,-2.348158,86326
122,כסיפה,1,-2.166536,21654
8,אום בטין,1,-2.139476,4371
264,שגב-שלום,1,-2.123988,13396
18,אל סייד,1,-2.069088,3407
213,ערערה-בנגב,1,-2.038620,21590


In [56]:
# @title Long Term care Rate vs. Socio-Economic Index in each settlement
df = data_master.copy()
cols = [
    'long_term_care_rate',
    'socio_economic_index_score',
    'peripherality_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset= cols)

fig = px.scatter(
    df,
    x='long_term_care_rate',
    y='socio_economic_index_score',
    hover_name='settlement_name',
  #  size='total_population',
  #  size_max=40,
    color='peripherality_index_cluster',
    color_continuous_scale='Plasma',
    trendline='ols',
    labels={
        'long_term_care_rate': 'Long Term care Rate (%)',
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'total_population': 'Population size',
        'peripherality_index_cluster': 'Peripherality Index'
    },
    title='Correlation Between Long Term care Rate and Socio-Economic Index'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between Long Term care Rate and socio economic index  is {data_master['long_term_care_rate'].corr(data_master['socio_economic_index_score'], method= 'spearman'):.2f}")

The correlation between Long Term care Rate and socio economic index  is -0.69


In [57]:
# @title Long Term care Rate vs. Income Support Rate in each settlement

df = data_master.copy()

cols = [
    'long_term_care_rate',
    'income_support_rate',
    'peripherality_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df[cols] = df[cols].astype(float)
df = df.dropna(subset= cols)

fig = px.scatter(
    df,
    x='long_term_care_rate',
    y='income_support_rate',
    hover_name='settlement_name',
  #  size='total_population',
  #  size_max=40,
    color='peripherality_index_cluster',
    color_continuous_scale='Plasma',
    trendline='ols',
    labels={
        'long_term_care_rate': 'Long Term care Rate (%)',
        'income_support_rate': 'Income Support Rate (%)',
        'total_population': 'Population size',
        'peripherality_index_cluster': 'Peripherality Index'
    },
    title='Correlation Between Long Term care Rate and Income Support Rate'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between Long Term care Rate and Income Support Rate is {data_master['long_term_care_rate'].corr(data_master['income_support_rate'], method= 'spearman'):.2f}")

The correlation between Long Term care Rate and Income Support Rate is 0.73


In [ ]:
df = data_master.copy()
corr = df[social_cols + health_cols].corr(method="spearman")
focused_corr = corr.loc[health_cols, social_cols]


fig = px.imshow(
    focused_corr,
    text_auto='.2f',
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    title='Spearman Correlations: Social vs Health Indicators'
)

fig.update_layout(
    xaxis_title='Social / Economic Indicators',
    yaxis_title='Health / Disability Indicators',
    title_font_size=18
)

fig.show()

The Intergenerational Trap:

In [59]:
# @title The Intergenerational Trap: Child vs. Adult Disability
# --- הכנה: חלוקה גיאוגרפית גסה (צפון, מרכז, דרום) ---
# נשתמש בקווי רוחב (Latitude) כדי לחלק את הארץ
def get_region(lat):
    if pd.isna(lat): return 'Unknown'
    if lat > 32.4: return 'North' # צפונית לחדרה בערך
    if lat < 31.7: return 'South' # דרומית לאשדוד/גדרה
    return 'Center'

data_master['region'] = data_master['lat'].apply(get_region)

# ==============================================================================
# תובנה 1: מלכודת הדורות (Intergenerational Trap)
# ==============================================================================
# סינון: רק יישובים עם אוכלוסייה משמעותית כדי למנוע רעש
df_trap = data_master.copy()
national_avg_adult_disability = (df_trap['general_disability_benefit'].sum() / df_trap['population_18_64'].sum()) * 100
national_avg_child_disability = (df_trap['disabled_child_benefit'].sum() / df_trap['population_0_17'].sum()) * 100
df_trap['socio_economic_index_cluster'] = pd.to_numeric(df_trap['socio_economic_index_cluster'], errors='coerce')

fig_trap = px.scatter(
    df_trap,
    x='general_disability_rate',
    y='disabled_child_benefit_rate',
    hover_name='settlement_name',
  #  size='total_population',
    color='socio_economic_index_cluster',
    color_continuous_scale='RdYlGn', # ירוק=טוב (אשכול גבוה), אדום=רע
    title='The Intergenerational Trap: Child vs. Adult Disability',
    labels={'general_disability_rate': 'Disability Benefit Recipient Rate (Ages 18-64) %',
            'disabled_child_benefit_rate': 'Child Disability Rate (Ages 0-17) %',
            'socio_economic_index_cluster': 'Socio-Economic Cluster'}
)

fig_trap.update_layout(
    shapes=[
        # קו אנכי: הממוצע הארצי של נכות מבוגרים
        dict(
            type="line",
            x0=national_avg_adult_disability,
            x1=national_avg_adult_disability,
            y0=0,
            y1=df_trap['disabled_child_benefit_rate'].max(),
            line=dict(color="black", dash="dash", width=2)
        ),

        # קו אופקי: הממוצע הארצי של נכות ילדים
        dict(
            type="line",
            x0=0,
            x1=df_trap['general_disability_rate'].max(),
            y0=national_avg_child_disability,
            y1=national_avg_child_disability,
            line=dict(color="black", dash="dash", width=2)
        ),

        # ריבוע אדום: אזור הסכנה (מעל הממוצע בשני המדדים)
        dict(
            type="rect",
            x0=national_avg_adult_disability,
            y0=national_avg_child_disability,
            x1=df_trap['general_disability_rate'].max(),
            y1=df_trap['disabled_child_benefit_rate'].max(),
            line=dict(color="Red", width=0), # ללא מסגרת
            fillcolor="Red",
            opacity=0.1,
            layer="below" # שהריבוע יהיה מתחת לנקודות
        )
    ]
)

# הוספת טקסט הסבר על הקווים
fig_trap.add_annotation(
    x=national_avg_adult_disability,
    y=df_trap['disabled_child_benefit_rate'].max(),
    text="National Avg (Adults)",
    showarrow=False,
    yshift=10
)

fig_trap.show()

print("--- 🚨 Red Alert: Towns with High Child AND Adult Disability ---")
high_child = df_trap['disabled_child_benefit_rate'] > df_trap['disabled_child_benefit_rate'].quantile(0.75)
high_adult = df_trap['general_disability_rate'] > df_trap['general_disability_rate'].quantile(0.75)
df_trap[high_child & high_adult][['settlement_name', 'region', 'socio_economic_index_cluster', 'disabled_child_benefit_rate', 'general_disability_rate']].sort_values('disabled_child_benefit_rate', ascending=False).head(10)

--- 🚨 Red Alert: Towns with High Child AND Adult Disability ---


,settlement_name,region,socio_economic_index_cluster,disabled_child_benefit_rate,general_disability_rate
246,קריית שמונה,North,5,8.84,11.06
97,טבריה,North,3,8.24,14.82
59,בת ים,Center,5,7.93,8.27
102,טירת כרמל,North,6,7.83,9.24
258,רמלה,Center,4,7.33,8.08
239,קריית גת,South,4,7.27,8.7
208,עפולה,North,5,7.19,11.22
94,חצור הגלילית,North,4,7.14,11.11
34,באר שבע,South,5,7.08,10.28
10,אור יהודה,Center,5,7.02,8.11


**The Data Story:**
This scatter plot reveals a disturbing correlation between **working-age disability** (parents) and **child disability** (future generation). The "Red Zone" (Top-Right Quadrant) represents localities where both rates exceed the national weighted average.

**Key Findings for Decision Makers:**
*   **The Vicious Cycle:** Disability in these areas is not an isolated individual event but a **household phenomenon**. Poverty and health issues are being transferred from generation to generation.
*   **Target Audience:** The localities in the top-right quadrant (marked by the red square) are the highest priority for intervention.
*   **Action Item:** Standard employment programs will likely fail here. These families require **holistic, multi-generational intervention** (treating the parent and child as a single unit) to break the cycle of dependency.

If there is a limited budget, it's better to put it on the 10 cities on this list.


<div class="alert alert-warning">
<h3>⚠️ Methodological Note: Administrative Data vs. Reality</h3>

**Critical Context for Decision Makers:**
The analysis below uses <b>Bituach Leumi administrative data</b>. It is crucial to distinguish between "Benefit Recipients" and the theoretical status of an individual:

*   <b>Unemployment Data:</b> Represents only those currently receiving <b>short-term unemployment benefits</b>. Long-term unemployed individuals who have exhausted their eligibility (common in distressed localities) are <b>NOT</b> counted in this specific denominator.
*   <b>Disability Data:</b> Represents those who successfully claimed benefits.

<b>Implication:</b> In the following graph ("The Welfare Trap"), a high ratio often indicates <b>structural, long-term unemployment</b> where residents have shifted from the temporary "unemployment track" to the permanent "disability track".
</div>

In [60]:
#@title The Welfare Trap: Chronic versus Temporary Benefits 
# ==============================================================================
#   (Chronic versus temporary benefits)תובנה 2: מסלול כרוני מול מסלול זמני
# ==============================================================================
df = data_master.copy()
# יחס נכות לאבטלה: כמה נכים יש על כל מובטל?
# יחס גבוה = ייתכן שאנשים "בורחים" לנכות כי אין עבודה, או שיש בעיה מבנית עמוקה
df['chronic_vs_temp_ratio'] = df['general_disability_benefit'] / df['unemployment_benefit']

# סינון רעשים  עם 0 מקבלי אבטלה)
df_ratio = df[df['unemployment_benefit'] > 10]

fig_ratio = px.bar(
    df_ratio.sort_values('chronic_vs_temp_ratio', ascending=False).head(15),
    x='settlement_name',
    y='chronic_vs_temp_ratio',
  #  color='region',
    title='The Welfare Trap: Ratio of Chronic (General Disability) to Temporary (Unemployment) Benefits',
    labels={'chronic_vs_temp_ratio': 'Ratio: Disability Recipients / Unemployment Recipients', 'settlement_name': 'Locality'},
    text_auto='.1f'
)
fig_ratio.update_layout(
    xaxis_tickangle=-45,
    # הוספת הערה בתחתית הגרף שמסבירה את המגבלה
    annotations=[dict(
        text="*Note: High ratio may indicate long-term unemployment where benefits have been exhausted.",
        x=0, y=-0.25, xref="paper", yref="paper", showarrow=False, font=dict(size=10, color="gray")
    )]
)
fig_ratio.show()


Localities with a high Disability-to-Unemployment ratio indicate a **collapsed labor market**. In these areas, disability benefits have become the de-facto social safety net instead of employment.

**Strategic Recommendations:**
1.  **Shift from Placement to Rehabilitation:** Standard employment bureaus are ineffective here. Resources should be diverted to **Vocational Rehabilitation Centers** that combine social work with soft-skills training.
2.  **"Laron Law" Optimization:** Launch targeted campaigns in these specific towns to educate residents on their right to work *without* losing their entire benefit (incentivizing part-time work).
3.  **Supply-Side Intervention:** Subsidies in these zones should focus exclusively on **"Inclusive Employers"** willing to adapt roles for individuals with chronic health issues.

---



In [61]:
# ==============================================================================
# תובנה 3: השוואה אזורית (צפון מול דרום)
# ==============================================================================

# נשווה רק יישובים מאשכולות נמוכים (1-4) כדי להשוות "תפוחים לתפוחים"
df_low_socio = data_master[data_master['socio_economic_index_cluster'].astype(float) <= 4].copy()

fig_region = px.box(
    df_low_socio,
    x='region',
    y='health_index',
    color='region',
    points='all', # מציג גם את הנקודות עצמן
    hover_name='settlement_name',
    title='Disability Rates in Low Socio-Economic Towns (Clusters 1-4): North vs. South vs. Center',
    category_orders={"region": ["North", "Center", "South"]}
)
fig_region.show()

**The Data Story:**
This analysis compares "apples to apples" (only low socio-economic settlements, clusters 1-4). The results are striking: **Geography dictates destiny.**
*   **The North (Blue):** Shows the highest median disability rate (~8%) and the most extreme outliers.
*   **The Center (Red):** Demonstrates a "protective effect" with significantly lower rates (~5%), even for equally poor populations.

**Key Takeaway for Decision Makers:**
Poverty in the periphery is "more toxic" than poverty in the center. The lack of accessible healthcare and employment infrastructure in the North exacerbates medical conditions.

**Action Item:**
**Prioritize the Northern District.** When allocating limited resources for health-employment interventions, a "poor town" in the North should receive higher weighting than a "poor town" in the Center.

In [62]:
#הכנה לגרף
## ---------- שמות עמודות ----------
DISABILITY_COUNT_COL = "general_disability_benefit"
WORKING_AGE_POP_COL = "population_18_64"
RATE_COL = "general_disability_rate"
SE_COL = "socio_economic_index_score"

## ---------- דאטה ----------
dfv = data_master.dropna(
    subset=[DISABILITY_COUNT_COL, WORKING_AGE_POP_COL, RATE_COL, SE_COL]
).copy()

dfv["disability_count"] = dfv[DISABILITY_COUNT_COL].astype(float)
dfv["pop_working_age"] = dfv[WORKING_AGE_POP_COL].astype(float)
dfv["disability_rate"] = dfv[RATE_COL].astype(float)
dfv["se_score"] = dfv[SE_COL].astype(float)

## ---------- רבעונים ----------
quartile_labels = ["Q1 (Lowest)", "Q2 (Low)", "Q3 (Medium)", "Q4 (Highest)"]
dfv["se_quartile"] = pd.qcut(dfv["se_score"], 4, labels=quartile_labels)

## ---------- עמודות: ממוצע שיעור נכות לרבעון (לא משוקלל) ----------
summary = (
    dfv.groupby("se_quartile", observed=True)
       .agg(
           avg_disability_rate=("disability_rate", "mean"),
           n=("disability_rate", "size")
       )
       .reset_index()
)

summary["se_quartile"] = pd.Categorical(
    summary["se_quartile"],
    categories=quartile_labels,
    ordered=True
)
summary = summary.sort_values("se_quartile")

## ---------- Benchmark: משוקלל לפי אוכלוסייה בגיל עבודה (18–64) ----------
overall_working_age_weighted = (
    dfv["disability_count"].sum() / dfv["pop_working_age"].sum()
) * 100

## ---------- צבעים ידניים לרבעונים ----------
color_map = {
    "Q1 (Lowest)": "#d73017",
    "Q2 (Low)": "#ff6e31",
    "Q3 (Medium)": "#dee00b",
    "Q4 (Highest)": "#1ac850",
}

# ---------- גרף (Plotly Express) ----------
fig = px.bar(
    summary,
    x="se_quartile",
    y="avg_disability_rate",
    color="se_quartile",
    color_discrete_map=color_map,
    text=summary["avg_disability_rate"].round(2),
    category_orders={"se_quartile": quartile_labels},
    labels={
        "se_quartile": "Socio-Economic Quartile",
        "avg_disability_rate": "Average Disability Rate (%)",
    },
    title="Disability Rates by Socio-Economic Quartile"
)

# hover עם מספר יישובים (גודל N)
fig.update_traces(
    textposition="outside",
    customdata=summary[["n"]].to_numpy(),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Avg disability rate (settlement mean): %{y:.2f}%<br>"
        "Number of settlements: %{customdata[0]}"
        "<extra></extra>"
    )
)

# ---------- עיצוב ----------
fig.update_layout(
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    width=650,
    bargap=0.3,
    yaxis=dict(
        range=[0, 10.1],
        showgrid=True,
        gridcolor="rgba(0,0,0,0.2)"
    ),
    xaxis=dict(showgrid=False),
    showlegend=False
)

# ---------- קו benchmark ----------
fig.add_hline(
    y=overall_working_age_weighted,
    line_width=3,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Overall disabilty rate: {overall_working_age_weighted:.2f}%",
    annotation_position="top left",
)

fig.show()

### What was done here and what it means

In this cell, we summarized the relationship between socio-economic status and disability **at the group level**, rather than at the level of individual settlements.

**What we did, briefly:**
- Divided settlements into four equal-sized quartiles based on the socio-economic index (from lowest to highest).
- For each quartile, calculated the **unweighted average** disability rate (each settlement counted equally).
- In addition, calculated a **national benchmark** weighted by the working-age population (18–64).
- Visualized the results using a bar chart, with a horizontal line representing the weighted national average.

**What the results show:**
- There is a **sharp, non-linear decline** in disability rates as socio-economic status increases.
- The largest differences appear between the lower socio-economic quartiles and the higher ones, indicating clear **high-risk zones** at the lower end of the socio-economic scale.
- The highest quartile shows substantially lower disability rates, well below the weighted national average, while the lower quartiles are clearly above it.
- Comparing unweighted quartile averages with the weighted national benchmark highlights that the burden of disability is concentrated in socio-economically weaker settlements, rather than being driven simply by population size.
```


In [ ]:
from scipy.stats import spearmanr

# -----------------------------
# הגדרת נקודת פיצול בין ערים גדולות  לבינוניות ויישובים קטנים
# -----------------------------
POP_CUTOFF = 40_000

# -----------------------------
# ניקוי נתונים
# -----------------------------
df = data_master.dropna(
    subset=[
        "general_disability_rate",
        "socio_economic_index_score",
        "peripherality_index_score",
        "total_population",
        "settlement_name",
    ]
).copy()

df["disability_rate"] = df["general_disability_rate"].astype(float)
df["se_score"] = df["socio_economic_index_score"].astype(float)
df["peripherality"] = df["peripherality_index_score"].astype(float)
df["population"] = df["total_population"].astype(float)

# פיצול לשני גרפים
df_large = df[df["population"] >= POP_CUTOFF].copy()
df_small = df[df["population"] < POP_CUTOFF].copy()

# -----------------------------
# פונקציה לשימוש בשני הגרפים
# -----------------------------
def make_fig(df_part, title_prefix, size_max):
    rho, p = spearmanr(df_part["disability_rate"], df_part["se_score"])
    sp_txt = f"Spearman ρ = {rho:.2f}, p = {p:.2e}"


    fig = px.scatter(
        df_part,
        x="disability_rate",
        y="se_score",
        color="peripherality",
        color_continuous_scale="Cividis",
        hover_name="settlement_name",
        size="population",
        size_max=size_max,
        trendline="lowess",
        title=f"{title_prefix}<br>{sp_txt}",
        labels={
            "disability_rate": "General Disability Rate (%)",
            "se_score": "Socio-Economic Index",
        },
    )

    # בלי מקרא צבעים
    fig.update_coloraxes(showscale=False)

    # hover לנקודות (עם אוכלוסייה ופריפריה)
    fig.update_traces(
        hovertemplate=(
            "<b>%{hovertext}</b><br>"
            "General Disability Rate: %{x:.2f}%<br>"
            "Socio-Economic Index: %{y:.2f}<br>"
            "Population: %{marker.size:,}<br>"
            "Peripherality Index: %{marker.color:.2f}"
            "<extra></extra>"
        ),
        selector=dict(mode="markers"),
    )

    # hover לקו הטרנד (רק שינוי טקסט)
    fig.update_traces(
        hovertemplate=(
            "Trendline<br>"
            "General Disability Rate: %{x:.2f}%<br>"
            "Socio-Economic Index: %{y:.2f}"
            "<extra></extra>"
        ),
        selector=dict(mode="lines"),
    )

    # עיצוב קו הטרנד
    fig.update_traces(line=dict(color="black", width=2), selector=dict(mode="lines"))
    # גודל הגרף
    fig.update_layout(template="plotly_white", width=780, height=490)

    return fig

# -----------------------------
# גרפים - שימוש בפונקציה
# -----------------------------
fig_large = make_fig(
    df_large,
    title_prefix=f"Large Settlements (≥{POP_CUTOFF:,})",
    size_max=35
)
fig_large.show()

fig_small = make_fig(
    df_small,
    title_prefix=f"Small & Medium Settlements (<{POP_CUTOFF:,})",
    size_max=12
)
fig_small.show()

### What was done in this cell

In this cell, we examined how **the strength of the relationship between the general disability rate and the socio-economic index varies by settlement size**.

To do so, settlements were divided into two groups:
- **Large cities** (more than 40,000 residents)
- **Small and medium-sized settlements** (fewer than 40,000 residents)

For each group, a separate scatter plot was produced and Spearman’s rank correlation was calculated.

The key finding is that the relationship is **stronger and more pronounced in small and medium-sized settlements**, while it is weaker in large cities. This indicates that the overall pattern observed in previous analyses is not driven by large cities; if anything, the opposite is true.


המחקר של גל:
סיכום המחקר: בחירות מתודולוגיות והיגיון אנליטי
מטרת המחקר

המחקר נועד לבחון את הטענה הרווחת בדבר קשר מעגלי בין עוני, פריפריאליות ומוגבלות, אך לא רק לשאול האם קיים קשר – אלא היכן הוא חזק במיוחד, ובאילו סוגי מוגבלות. המטרה היישומית הייתה לזהות מוקדים בהם ניתן למקד התערבות חברתית-טכנולוגית אפקטיבית.

בחירת יחידת הניתוח: יישוב

המחקר מתבצע ברמת היישוב, ולא הפרט, מתוך הנחה שמוגבלות היא תופעה:

מרחבית

מבנית

מושפעת מהקשר חברתי-כלכלי

בחירה זו מאפשרת:

שילוב מדדים סוציו-אקונומיים, גיאוגרפיים ובריאותיים

זיהוי דפוסים אזוריים וחריגים יישוביים

תרגום ממצאים למדיניות ולהקצאת משאבים

למה לא הסתפקתי במספר מקבלי קצבאות?

מספרים מוחלטים של מקבלי קצבאות מושפעים ישירות מגודל היישוב. לכן, שימוש בהם היה:

מטעה

מחזק יישובים גדולים באופן מלאכותי

מונע השוואה אמיתית בין יישובים

הבחירה המרכזית: חישוב שיעורי תחלואה (Rates)
איך חישבתי את השיעורים?

לכל סוג קצבה חושב שיעור כך:

שיעור=מספר מקבלי קצבהאוכלוסיית היישוב הרלוונטית×1,000
שיעור=
אוכלוסיית היישוב הרלוונטית
מספר מקבלי קצבה
	​

×1,000

(או לכלל האוכלוסייה, בהתאם לסוג הקצבה)

למה חישוב שיעור ולא מספר?

המעבר משימוש במספרים מוחלטים לשיעורים:

מנטרל את גודל היישוב

מאפשר השוואה בין יישובים קטנים וגדולים

חושף פערים מבניים אמיתיים

כלומר:

השאלה אינה “איפה יש יותר נכים”, אלא
“איפה הסיכון להיות נכה גבוה יותר”

בידול בין סוגי מוגבלות – החלטה קריטית

במקום להתייחס ל”נכות” כמקשה אחת, בוצעה הבחנה בין סוגי קצבאות, מתוך הנחה שלכל סוג:

מנגנון סיבתי שונה

קשר שונה לעוני

שלוש קטגוריות מרכזיות שנבחנו:
1. נפגעי עבודה

נבחרו כקבוצת ביקורת, משום שהם:

נובעים מאירוע תעסוקתי

אינם תוצאה של תהליך עוני מצטבר

צפויים להיות פחות תלויים במצב סוציו-אקונומי

2. קצבאות סיעוד (זקנה)

נכללו בניתוח נפרד משום שהן:

תלויות מאוד בגיל

מושפעות גם מגורמים ביולוגיים טבעיים

עלולות “לזהם” ניתוח של נכות הקשורה לעוני

3. נכות כללית “קשורה לעוני”

נבנתה מדד אגרגטיבי הכולל:

נכות כללית

שירותים מיוחדים

הבטחת הכנסה

והוחרגו ממנו:

נפגעי עבודה

קצבאות זקנה

כך נוצר מדד שמייצג מוגבלות בגיל העבודה, עם תלות כלכלית – כלומר נכות שיש לה פוטנציאל להנציח עוני.

למה חישוב נפרד לפי גיל?

הניתוח בודד קבוצות גיל (בעיקר 18–64) כדי:

למנוע הטיה של יישובים “זקנים”

להתמקד באוכלוסייה שבה נכות משפיעה ישירות על השתתפות בשוק העבודה

לחדד את הקשר בין נכות, תעסוקה ועוני

בחינת הקשר ל-SEI

לאחר חישוב שיעורי התחלואה:

נבדק הקשר בין כל שיעור לבין מדד סוציו-אקונומי (SEI)

הוצגו גרפים עם:

קו מגמה

גודל נקודה לפי אוכלוסייה

צביעה לפי אשכול SEI

חושבו מדדי קורלציה (Pearson, Spearman)

למה ניתוח חריגים?

מעבר לממוצע:

זוהו יישובים החורגים מהקשר הכללי

בוצע ניתוח איכותני של חריגים “טובים” ו”רעים”

המטרה:

להבין אילו מנגנונים מחזקים או שוברים את הקשר

לאתר מוקדי התערבות והעתקה (POC)

המסר המרכזי של המתודולוגיה

המחקר אינו בוחן “כמה נכים יש”, אלא
איפה, באיזה הקשר חברתי, ובאיזה סוג מוגבלות – הסיכון לנכות גבוה במיוחד.

הבחירה לעבוד עם שיעורים, בידול סוגי קצבאות, ובידוד גילי עבודה היא שמאפשרת לעבור מגרף יפה לניתוח שמסוגל להנחות פעולה.

In [ ]:
# ====== התאמות נדרשות: החלף שמות עמודות אם שונים בדאטה שלך ======
COL_TOTAL_POP = 'total_population'
COL_SEI = 'socio_economic_index_score'
COL_PERIPH = 'peripherality_index_score'
COL_GENERAL = 'general_disability_benefit'
COL_LONGTERM = 'long_term_care_benefit'
COL_WORK_INJ = "work_injury_victims_receiving_disability_and_dependents’_benefits"  # בדוק שם מדויק
COL_POP_0_17 = 'population_0_17'
COL_POP_18_64 = 'population_18_64'
COL_POP_65 = 'population_65_plus'
# ===================================================================

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px # Added import for plotly

# df = pd.read_csv(...)  # טען כאן את ה-DataFrame שלך
# ---------------------------------------------------------------------------------
# 1) הכנה ונירמול לשיעורים per1000
# ---------------------------------------------------------------------------------
def safe_numeric(df, col):
    return pd.to_numeric(df[col], errors='coerce')

# המרת עמודות למספרים (אם לא כבר)
for c in [COL_TOTAL_POP, COL_SEI, COL_PERIPH, COL_GENERAL, COL_LONGTERM, COL_WORK_INJ,
          COL_POP_0_17, COL_POP_18_64, COL_POP_65]:
    if c in df.columns:
        df[c] = safe_numeric(df, c)

# Drop שורות בלי אוכלוסיה תקינה
df = df[df[COL_TOTAL_POP].notna() & (df[COL_TOTAL_POP] > 0)].copy()

# מחשבים שיעורים per1000 לכל קצבה קיימת
benefit_cols = [COL_GENERAL, COL_LONGTERM, COL_WORK_INJ]
for c in benefit_cols:
    if c in df.columns:
        df[c + '_per1000'] = df[c] / df[COL_TOTAL_POP] * 1000

# שיעורים לפי גיל (אם יש נתונים)
if COL_POP_18_64 in df.columns:
    df['rate_18_64_per1000'] = (df[COL_GENERAL] / df[COL_POP_18_64]) * 1000
if COL_POP_65 in df.columns:
    df['rate_65_plus_per1000'] = (df[COL_GENERAL] / df[COL_POP_65]) * 1000

# ---------------------------------------------------------------------------------
# 2) שיעור אגרגטיבי (aggregate disability rate)
# שים לב: אם קיימת חפיפה בין קטגוריות יש צורך לבצע de-duplication (רק אם יש מזהה יחידני).
# כאן נעשה אגרגציה פשוטה אך שמרנית: ניקח max בין סכום לבין כלל, כדי לא להקטין.
# ---------------------------------------------------------------------------------
# אופציה א: aggregate_by_sum = sum של כל הקצבאות (אם אין כפל)
df['agg_count_sum'] = 0
for c in benefit_cols:
    if c in df.columns:
        df['agg_count_sum'] = df['agg_count_sum'] + df[c].fillna(0)
df['agg_rate_sum_per1000'] = df['agg_count_sum'] / df[COL_TOTAL_POP] * 1000

# אופציה ב (שמרנית): השתמש ב-general כייצוג בסיסי, אלא אם סה"כ גדול יותר
df['agg_rate_per1000'] = df['agg_rate_sum_per1000']
if COL_GENERAL in df.columns:
    df.loc[df['agg_rate_sum_per1000'] < (df[COL_GENERAL+'_per1000'].fillna(0)), 'agg_rate_per1000'] = df[COL_GENERAL+'_per1000']

# ---------------------------------------------------------------------------------
# 3) הגדרת "מוגבלויות הקשורות לעוני" (non-age, non-accident)
# ---------------------------------------------------------------------------------
# ניסוח בסיסי:
# poverty_related_count = general - longterm - work_injury
# אם יש שירותים מיוחדים שנחשבים לעוני (לדוגמה: special_services_for_persons_with_disabilities) תוכל לכלול
LONGTERM = COL_LONGTERM in df.columns
WORKINJ = COL_WORK_INJ in df.columns

df['poverty_related_count'] = df[COL_GENERAL].fillna(0)
if LONGTERM:
    df['poverty_related_count'] = df['poverty_related_count'] - df[COL_LONGTERM].fillna(0)
if WORKINJ:
    df['poverty_related_count'] = df['poverty_related_count'] - df[COL_WORK_INJ].fillna(0)
# לא לרדת מתחת לאפס
df['poverty_related_count'] = df['poverty_related_count'].clip(lower=0)
df['poverty_related_rate_per1000'] = df['poverty_related_count'] / df[COL_TOTAL_POP] * 1000

# ---------------------------------------------------------------------------------
# 4) משתנים מתערבים מומלצים לחסימה
# ---------------------------------------------------------------------------------
# תמיד לכלול: log(pop), %65+, peripherality, אם יש - unemployment, income_support etc.
df['log_pop'] = np.log(df[COL_TOTAL_POP])
if COL_POP_65 in df.columns:
    df['pct_65_plus'] = df[COL_POP_65] / df[COL_TOTAL_POP] * 100
else:
    # אם אין 65+, נסה לחשב מ־0_17 ו־18_64 אם יש
    if (COL_POP_0_17 in df.columns) and (COL_POP_18_64 in df.columns):
        df['population_65_plus'] = df[COL_TOTAL_POP] - df[COL_POP_0_17] - df[COL_POP_18_64]
        df['pct_65_plus'] = df['population_65_plus'] / df[COL_TOTAL_POP] * 100

# אם יש מדדי תעסוקה/תמיכה כלכלית
# COL_INCOME_SUPPORT = 'income_support_benefit'  # אם יש, המר והוסף ל-list של Confounders

# ---------------------------------------------------------------------------------
# 5) בדיקות קורלציה בסיסיות: Pearson + Spearman בין SEI ל־poverty_related_rate_per1000
# ומדידה של קורלציה חלקית תוך שליטה בלוג־אוכלוסייה ו-%65+
# ---------------------------------------------------------------------------------
target = 'poverty_related_rate_per1000'
if target not in df.columns or COL_SEI not in df.columns:
    raise ValueError("חסרים העמודות target או SEI. בדוק שמות העמודות והתאמן אותם בקוד.")

# Ensure target and control columns are numeric (float64) for statsmodels
df[target] = df[target].astype(float)
df[COL_SEI] = df[COL_SEI].astype(float)

# Pearson & Spearman (pairwise)
x = df[COL_SEI].dropna()
y = df.loc[x.index, target]
pearson_r, pearson_p = stats.pearsonr(x, y)
spearman_r, spearman_p = stats.spearmanr(x, y)

print("Pearson r (SEI vs poverty-related rate): r={:.3f}, p={:.3e}".format(pearson_r, pearson_p))
print("Spearman rho: rho={:.3f}, p={:.3e}".format(spearman_r, spearman_p))

# Partial correlation controlling for log_pop and pct_65_plus (regress-out residuals)
controls = ['log_pop']
if 'pct_65_plus' in df.columns:
    controls.append('pct_65_plus')

# make sure control cols exist and no NaN problem and convert to float
control_df = df[controls].astype(float)
valid_idx = df[[COL_SEI, target]].dropna().index.intersection(control_df.dropna().index)

res_y = sm.OLS(df.loc[valid_idx, target], sm.add_constant(control_df.loc[valid_idx, controls])).fit().resid
res_x = sm.OLS(df.loc[valid_idx, COL_SEI], sm.add_constant(control_df.loc[valid_idx, controls])).fit().resid
partial_r, partial_p = stats.pearsonr(res_x, res_y)
print("Partial correlation controlling for {}: r={:.3f}, p={:.3e}".format(controls, partial_r, partial_p))

# ---------------------------------------------------------------------------------
# 6) רגרסיה מרובת משתנים לחיזוק המסקנות (robust SE)
# ---------------------------------------------------------------------------------
predictors = [COL_SEI, 'log_pop']
if 'pct_65_plus' in df.columns:
    predictors.append('pct_65_plus')
if COL_PERIPH in df.columns:
    predictors.append(COL_PERIPH)

# Ensure predictor columns are numeric (float64)
X_reg = df[predictors].astype(float)

X = X_reg.dropna()
y_reg = df.loc[X.index, target].astype(float)
Xc = sm.add_constant(X)
model = sm.OLS(y_reg, Xc).fit(cov_type='HC3')  # robust SE
print(model.summary())

# תוספת: הפעל אינטראקציה SEI x peripherality אם שניהם קיימים
if (COL_SEI in df.columns) and (COL_PERIPH in df.columns):
    df['SEI_x_PER'] = df[COL_SEI] * df[COL_PERIPH]
    preds = [COL_SEI, COL_PERIPH, 'SEI_x_PER', 'log_pop']
    if 'pct_65_plus' in df.columns: preds.append('pct_65_plus')

    X2_reg = df[preds].astype(float)
    X2 = X2_reg.dropna()
    y2 = df.loc[X2.index, target].astype(float)
    X2c = sm.add_constant(X2)
    model2 = sm.OLS(y2, X2c).fit(cov_type='HC3')
    print("\nInteraction model summary (SEI x Peripherality):\n")
    print(model2.summary())

# ---------------------------------------------------------------------------------
# 7) זיהוי חריגים (יישובים שבהם observed >> predicted)
# ---------------------------------------------------------------------------------
# השתמש במודל האחרון שיצרת (או במודל בסיסי) כדי לחשב שאריות
if 'model' in locals():
    df.loc[X.index, 'predicted'] = model.predict(Xc)
    df['residual'] = df[target] - df['predicted']
    df['residual_std'] = (df['residual'] - df['residual'].mean()) / df['residual'].std()
    outliers = df.sort_values('residual', ascending=False).head(30)[['settlement_name', COL_TOTAL_POP, target, 'predicted','residual','residual_std']]
    print("\nTop positive residual outliers (likely institutional or hotspots):")
    display(outliers)

# ---------------------------------------------------------------------------------
# 8) גרפים להמחשה (scatter + residuals histogram)
# ---------------------------------------------------------------------------------

# Scatter plot: SEI vs poverty-related rate
plot_df = df.loc[X.index].copy() # Use the same filtered data as the regression model
# Ensure relevant columns are numeric for plotly.express color/size
plot_df['socio_economic_index_cluster'] = pd.to_numeric(plot_df['socio_economic_index_cluster'], errors='coerce')
plot_df['total_population'] = pd.to_numeric(plot_df['total_population'], errors='coerce')

fig_scatter = px.scatter(
    plot_df.dropna(subset=['socio_economic_index_score', 'poverty_related_rate_per1000', 'socio_economic_index_cluster', 'total_population']),
    x='socio_economic_index_score',
    y='poverty_related_rate_per1000',
    hover_name='settlement_name',
    size='total_population',
    size_max=60,
    color='socio_economic_index_cluster',
    color_continuous_scale='Viridis', # Or any other suitable scale
    trendline='ols',
    labels={
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'poverty_related_rate_per1000': 'Poverty-Related Disability Rate (per 1000)',
        'total_population': 'Population size',
        'socio_economic_index_cluster': 'Socio-Economic Index Cluster'
    },
    title='Socio-Economic Index vs. Poverty-Related Disability Rate'
)
fig_scatter.update_layout(title_font_size=18, xaxis_title_font_size=14, yaxis_title_font_size=14)
fig_scatter.show()

# Residuals histogram
if 'residual' in df.columns:
    fig_hist = px.histogram(
        df.loc[X.index].dropna(subset=['residual']), # Use filtered data with residuals
        x='residual',
        nbins=40,
        title='Distribution of Residuals (Observed - Predicted)'
    )
    fig_hist.update_layout(title_font_size=18, xaxis_title_font_size=14, yaxis_title_font_size=14)
    fig_hist.show()

Pearson r (SEI vs poverty-related rate): r=-0.490, p=3.546e-18
Spearman rho: rho=-0.475, p=4.753e-17
Partial correlation controlling for ['log_pop', 'pct_65_plus']: r=-0.360, p=6.244e-10
                                 OLS Regression Results                                 
Dep. Variable:     poverty_related_rate_per1000   R-squared:                       0.283
Model:                                      OLS   Adj. R-squared:                  0.273
Method:                           Least Squares   F-statistic:                     35.09
Date:                          Thu, 15 Jan 2026   Prob (F-statistic):           1.20e-23
Time:                                  11:17:45   Log-Likelihood:                -899.79
No. Observations:                           278   AIC:                             1810.
Df Residuals:                               273   BIC:                             1828.
Df Model:                                     4                                         
Covariance T

,settlement_name,total_population,poverty_related_rate_per1000,predicted,residual,residual_std
106,יבנאל,4502,36.428254,8.858407,27.569847,4.469240
63,ג'סר א-זרקא,15778,36.379769,9.110475,27.269295,4.420519
150,מגדל,2076,29.865125,5.280574,24.584552,3.985306
171,מתתיהו,3437,33.168461,8.993267,24.175194,3.918946
227,צפת,39301,26.665988,7.373631,19.292357,3.127409
97,טבריה,52388,23.955868,4.935054,19.020813,3.083390
75,גני מודיעין,3084,23.994812,7.882382,16.112430,2.611923
116,ירכא,17188,21.875727,6.588535,15.287192,2.478147
98,טובא-זנגרייה,6905,22.013034,8.121175,13.891859,2.251955
3,אבטין,2931,23.541453,9.908854,13.632600,2.209928


In [ ]:
df_senior = data_master.copy()

cols_for_plot = [
    'long_term_care_rate',
    'socio_economic_index_score',
    'population_65_plus',
    'settlement_name',
    'socio_economic_index_cluster'
]

# Ensure relevant columns are numeric, coercing errors to NaN
# Then explicitly convert ALL to standard float types (float64)
for col in ['long_term_care_rate', 'socio_economic_index_score', 'population_65_plus', 'socio_economic_index_cluster']:
    df_senior[col] = pd.to_numeric(df_senior[col], errors='coerce').astype(float)

# Drop rows with NaNs in any of the columns critical for plotting
# Now that they are all float64, dropna will work cleanly
df_senior = df_senior.dropna(subset=['long_term_care_rate', 'socio_economic_index_score', 'population_65_plus', 'socio_economic_index_cluster'])

fig = px.scatter(
    df_senior,
    x='socio_economic_index_score',
    y='long_term_care_rate',
    hover_name='settlement_name',
    size='population_65_plus',
    size_max=60,
    color='socio_economic_index_cluster',
    color_continuous_scale='Viridis',
    trendline='ols',
    labels={
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'long_term_care_rate': 'Long-Term Care Rate (per 100 seniors)',
        'population_65_plus': 'Population 65+',
        'socio_economic_index_cluster': 'Socio-Economic Index Cluster'
    },
    title='Correlation: Senior Long-Term Care Rate vs. Socio-Economic Index',
    width=1400,
    height=750
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()

In [ ]:
import plotly.express as px

COL_SEI = 'socio_economic_index_score'
COL_WORK_INJ = "work_injury_victims_receiving_disability_and_dependents’_benefits"
COL_TOTAL_POP = 'population_18_64'
COL_SEI_CLUSTER = 'socio_economic_index_cluster'

df_plot = data_master.copy()

# Ensure relevant columns are numeric and handle NaNs for plotting
for col in [COL_SEI, COL_WORK_INJ, COL_TOTAL_POP, COL_SEI_CLUSTER]:
    df_plot[col] = pd.to_numeric(df_plot[col], errors='coerce').astype(float)

# Calculate work_injury_rate_per_1000 *before* comprehensive filtering
df_plot['work_injury_rate_per1000'] = (
    df_plot[COL_WORK_INJ] / df_plot[COL_TOTAL_POP] * 1000
)

# Filter out rows with missing values after conversion, and ensure total_population is positive
# Also filter on the newly created rate column
df_plot = df_plot[
    df_plot[COL_SEI].notna() &
    df_plot[COL_WORK_INJ].notna() & # Keep this if COL_WORK_INJ itself is still needed for context
    df_plot[COL_TOTAL_POP].notna() &
    (df_plot[COL_TOTAL_POP] > 0) &
    df_plot['work_injury_rate_per1000'].notna() # Ensure the rate itself is not NaN
].copy()

# Create the interactive scatter plot
fig = px.scatter(
    df_plot,
    x=COL_SEI,
    y='work_injury_rate_per1000',
    hover_name='settlement_name',
    size=COL_TOTAL_POP,
    size_max=60, # Max size for the bubbles
    color=COL_SEI_CLUSTER,
    color_continuous_scale='Magma', # Changed color scale for variety
    trendline='ols',
    labels={
        COL_SEI: 'Socio-Economic Index Score',
        'work_injury_rate_per1000': 'Work Injury Disability Rate (per 1,000)',
        COL_TOTAL_POP: 'Total Population',
        COL_SEI_CLUSTER: 'Socio-Economic Index Cluster'
    },
    title='Socio-Economic Index vs. Work-Injury Disability Rate',
    width=1400,
    height=750
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()

# Recalculate correlations on the cleaned data for consistency with the plot
x_corr = df_plot[COL_SEI]
y_corr = df_plot['work_injury_rate_per1000']

pearson_r, pearson_p = stats.pearsonr(x_corr, y_corr)
spearman_r, spearman_p = stats.spearmanr(x_corr, y_corr)

print(f"Pearson r: {pearson_r:.3f} (p={pearson_p:.3e})")
print(f"Spearman rho: {spearman_r:.3f} (p={spearman_p:.3e})")

Pearson r: 0.140 (p=1.965e-02)
Spearman rho: 0.127 (p=3.391e-02)


# Task
The user wants to identify and visualize 'bad' and 'good' outlier settlements based on their socio-economic index score and poverty-related disability rate.

To achieve this, I will perform the following steps:
1.  **Prepare Data for Outlier Analysis**: I will create a new DataFrame `df_outliers` from `data_master`, including `socio_economic_index_score`, `poverty_related_rate_per1000`, `socio_economic_index_cluster`, `settlement_name`, and `total_population`. I will ensure all numeric columns are correctly typed and handle any missing values by dropping rows where these key metrics are NaN, which is crucial for accurate percentile calculations and outlier identification.
2.  **Define Outlier Thresholds**: I will calculate the 25th (Q1) and 75th (Q3) percentiles for `socio_economic_index_score` and `poverty_related_rate_per1000`. These percentiles will serve as thresholds to classify settlements.
3.  **Identify and Classify Outliers**: I will create a new column, `Outlier_Type`, in `df_outliers`.
    *   A settlement will be classified as a **'Bad Outlier'** if it has a low `socio_economic_index_score` (below Q1) AND a high `poverty_related_rate_per1000` (above Q3). These represent highly disadvantaged settlements with a high burden of poverty-related disability.
    *   A settlement will be classified as a **'Good Outlier'** if it has a low `socio_economic_index_score` (below Q1) AND a low `poverty_related_rate_per1000` (below Q1). These represent disadvantaged settlements that are performing relatively well in terms of poverty-related disability rates.
    *   All other settlements will be labeled **'Normal'**.
4.  **Examine Outlier Settlements**: I will display the top 'Bad Outlier' and 'Good Outlier' settlements, showing their `settlement_name`, `socio_economic_index_score`, `poverty_related_rate_per1000`, `socio_economic_index_cluster`, and `total_population`.
5.  **Visualize Outliers**: I will generate an interactive scatter plot using `plotly.express`. The x-axis will represent `socio_economic_index_score` and the y-axis `poverty_related_rate_per1000`. Points will be colored based on their `Outlier_Type` (red for 'Bad Outlier', green for 'Good Outlier', blue for 'Normal'), and their size will reflect `total_population` for additional context. Hover functionality will display settlement details.
6.  **Final Task**: I will summarize the characteristics of the identified outlier settlements and discuss the insights gained from their classification and visualization, addressing the initial query about understanding 'bad' and 'good' outliers and their comparison within socio-economic clusters.

This comprehensive approach will allow us to pinpoint settlements that deviate significantly from expected patterns based on their socio-economic standing and disability rates.

## Prepare Data for Outlier Analysis

### Subtask:
Ensure 'socio_economic_index_score', 'poverty_related_rate_per1000', 'socio_economic_index_cluster', 'settlement_name', and 'total_population' columns in `data_master` are numeric and handle any missing values, preparing the data for outlier identification.


**Reasoning**:
The subtask requires several data preparation steps: calculating new rates, creating a working DataFrame copy, converting specified columns to numeric types, and handling missing values in those columns. Combining these into a single code block ensures efficient data manipulation.



In [ ]:
data_master['poverty_related_disability'] = (
    data_master['general_disability_benefit'].fillna(0) +
    data_master['special_services_for_persons_with_disabilities'].fillna(0)
)

data_master['poverty_related_rate_per1000'] = (
    data_master['poverty_related_disability'] /
    data_master['population_18_64']
) * 1000

df_outliers = data_master.copy()

cols_for_outliers = [
    'socio_economic_index_score',
    'poverty_related_rate_per1000',
    'socio_economic_index_cluster',
    'settlement_name',
    'total_population'
]

for col in cols_for_outliers:
    if col != 'settlement_name': # settlement_name is string, no need to convert to numeric
        df_outliers[col] = pd.to_numeric(df_outliers[col], errors='coerce')

df_outliers = df_outliers.dropna(subset=cols_for_outliers).copy()

## Define Outlier Thresholds

### Subtask:
Calculate the 25th and 75th percentiles for 'socio_economic_index_score' and 'poverty_related_rate_per1000' to establish criteria for 'low'/'high' values for outlier identification.


**Reasoning**:
To define outlier thresholds, I will calculate the 25th and 75th percentiles for 'socio_economic_index_score' and 'poverty_related_rate_per1000' from the `df_outliers` DataFrame.



In [ ]:
sei_q1 = df_outliers['socio_economic_index_score'].quantile(0.25)
sei_q3 = df_outliers['socio_economic_index_score'].quantile(0.75)
poverty_rate_q1 = df_outliers['poverty_related_rate_per1000'].quantile(0.25)
poverty_rate_q3 = df_outliers['poverty_related_rate_per1000'].quantile(0.75)

print(f"Socio-Economic Index Score Q1: {sei_q1:.2f}")
print(f"Socio-Economic Index Score Q3: {sei_q3:.2f}")
print(f"Poverty-Related Rate per 1000 Q1: {poverty_rate_q1:.2f}")
print(f"Poverty-Related Rate per 1000 Q3: {poverty_rate_q3:.2f}")

Socio-Economic Index Score Q1: -0.78
Socio-Economic Index Score Q3: 0.73
Poverty-Related Rate per 1000 Q1: 52.22
Poverty-Related Rate per 1000 Q3: 99.74


## Identify and Classify Outliers

### Subtask:
Create a new column 'Outlier_Type' in `df_outliers`. Classify settlements as 'Bad Outlier' (low SEI score and high disability-to-poverty rate), 'Good Outlier' (low SEI score and low disability-to-poverty rate), or 'Normal' based on the defined thresholds.


**Reasoning**:
I will classify settlements into 'Bad Outlier', 'Good Outlier', or 'Normal' categories in the 'Outlier_Type' column of `df_outliers` based on the specified conditions and previously calculated quartile thresholds.



In [ ]:
df_outliers['Outlier_Type'] = 'Normal'
df_outliers.loc[
    (df_outliers['socio_economic_index_score'] < sei_q1) &
    (df_outliers['poverty_related_rate_per1000'] > poverty_rate_q3),
    'Outlier_Type'
] = 'Bad Outlier'
df_outliers.loc[
    (df_outliers['socio_economic_index_score'] < sei_q1) &
    (df_outliers['poverty_related_rate_per1000'] < poverty_rate_q1),
    'Outlier_Type'
] = 'Good Outlier'

print("Settlements classified into Outlier_Type column.")

Settlements classified into Outlier_Type column.


## Examine Outlier Settlements

### Subtask:
Display the top 'Bad Outlier' and 'Good Outlier' settlements, showing their `settlement_name`, `socio_economic_index_score`, `poverty_related_rate_per1000`, `socio_economic_index_cluster`, and `total_population`.



**Reasoning**:
To examine the outlier settlements, I will filter `df_outliers` for 'Bad Outlier' and 'Good Outlier' types, sort them by relevant metrics, and display the top 10 settlements for each category with the requested columns.



In [ ]:
print("Top 10 Bad Outlier Settlements:")
df_bad_outliers = df_outliers[df_outliers['Outlier_Type'] == 'Bad Outlier'].sort_values(
    by=['socio_economic_index_score', 'poverty_related_rate_per1000'], ascending=[True, False]
).head(10)
display(df_bad_outliers[cols_for_outliers])

print("\nTop 10 Good Outlier Settlements:")
df_good_outliers = df_outliers[df_outliers['Outlier_Type'] == 'Good Outlier'].sort_values(
    by=['socio_economic_index_score', 'poverty_related_rate_per1000'], ascending=[True, True]
).head(10)
display(df_good_outliers[cols_for_outliers])

Top 10 Bad Outlier Settlements:


,socio_economic_index_score,poverty_related_rate_per1000,socio_economic_index_cluster,settlement_name,total_population
134,-1.452973,100.559076,2,כפר מנדא,22269
63,-1.447552,127.597717,2,ג'סר א-זרקא,15778
196,-1.410061,107.629428,2,עוזייר,3641
199,-1.404368,105.720492,2,עילוט,9163
3,-1.363299,127.671541,2,אבטין,2931
227,-1.350900,144.113563,2,צפת,39301
106,-1.236029,153.247941,2,יבנאל,4502
37,-1.203165,108.002992,2,ביר אל-מכסור,11104
169,-1.126993,104.58891,3,משהד,8900
200,-1.102853,100.456621,3,עין מאהל,14404



Top 10 Good Outlier Settlements:


,socio_economic_index_score,poverty_related_rate_per1000,socio_economic_index_cluster,settlement_name,total_population
233,-2.409800,48.085485,1,קצר א-סר,2704
242,-1.174201,49.445005,2,קריית יערים,7036
201,-1.153344,51.010587,2,עין נקובא,3799
111,-0.982367,37.714286,3,יצהר,2426


**Reasoning**:
Now that the outlier settlements have been identified and examined, I will create an interactive scatter plot to visualize the 'Bad Outlier', 'Good Outlier', and 'Normal' settlements based on their socio-economic index score and poverty-related disability rate, as described in the overall task.



In [ ]:
color_map = {
    'Bad Outlier': 'red',
    'Good Outlier': 'green',
    'Normal': 'blue'
}

fig = px.scatter(
    df_outliers,
    x='socio_economic_index_score',
    y='poverty_related_rate_per1000',
    hover_name='settlement_name',
    size='total_population',
    size_max=60,
    color='Outlier_Type',
    color_discrete_map=color_map,
    labels={
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'poverty_related_rate_per1000': 'Poverty-Related Disability Rate (per 1,000)',
        'total_population': 'Population size',
        'Outlier_Type': 'Outlier Classification'
    },
    title='Socio-Economic Index vs. Poverty-Related Disability Rate with Outliers',
    width=1400,
    height=750
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()

## Summary of Outlier Settlements

The analysis successfully identified settlements that are 'Bad Outliers' and 'Good Outliers' based on their socio-economic index score and poverty-related disability rates.

### Bad Outliers:
These settlements are characterized by a low socio-economic index score (indicating disadvantage) and a high poverty-related disability rate. The displayed 'Bad Outliers' typically show:
- **Low Socio-Economic Index Score**: Consistently below the 25th percentile (`sei_q1`).
- **High Poverty-Related Disability Rate**: Consistently above the 75th percentile (`poverty_rate_q3`).
- **Demographics**: Many of these settlements are smaller to medium-sized, often with lower socio-economic clusters (e.g., cluster 2 or 3).

These settlements represent communities facing significant dual challenges: deep socio-economic disadvantage coupled with a disproportionately high burden of poverty-related disabilities. This suggests a potential vicious cycle where poverty exacerbates disability, and disability perpetuates poverty within these communities. They could be targets for comprehensive interventions aimed at both socio-economic upliftment and disability support.

### Good Outliers:
These settlements are also characterized by a low socio-economic index score (indicating disadvantage), but they exhibit a surprisingly low poverty-related disability rate. The displayed 'Good Outliers' typically show:
- **Low Socio-Economic Index Score**: Consistently below the 25th percentile (`sei_q1`).
- **Low Poverty-Related Disability Rate**: Consistently below the 25th percentile (`poverty_rate_q1`).
- **Demographics**: These can vary, but like 'Bad Outliers', they tend to be on the lower end of the socio-economic cluster scale.

'Good Outliers' are particularly interesting. Despite facing similar socio-economic challenges as 'Bad Outliers', they manage to maintain lower rates of poverty-related disability. This raises important questions about what factors contribute to their relative resilience. These could include:
- **Effective Local Programs**: Local initiatives or community support systems that mitigate the impact of poverty on disability.
- **Demographic Nuances**: Specific demographic compositions (e.g., strong family support structures, cultural factors) that are not captured by the general socio-economic index.
- **Data Limitations**: Potential underreporting of disabilities or differences in access to benefits.

Studying these 'Good Outliers' could provide valuable insights into protective factors and successful strategies that could potentially be replicated in 'Bad Outlier' communities.

### Insights from Visualization:
The scatter plot visually reinforces these classifications. 'Bad Outliers' cluster in the bottom-right (low SEI score, high disability rate), while 'Good Outliers' appear in the bottom-left (low SEI score, low disability rate). The majority of 'Normal' settlements fall within the expected range, generally showing an inverse relationship between socio-economic standing and disability rates, with better-off communities (higher SEI score) tending to have lower poverty-related disability rates.

This analysis highlights the importance of looking beyond just overall socio-economic indicators and delving into specific outcomes like disability rates to identify communities that are either struggling more than expected or performing surprisingly well, thus informing targeted policy interventions.

## Final Task

### Subtask:
Summarize the characteristics of the identified outlier settlements and discuss the insights gained from their classification and visualization, addressing the initial query about understanding 'bad' and 'good' outliers and their comparison within socio-economic clusters.


## Summary:

### Q&A

**What constitutes a 'bad' outlier settlement?**
A 'bad' outlier settlement is defined as one with a low socio-economic index score (below the 25th percentile, which is -0.78) and a high poverty-related disability rate (above the 75th percentile, which is 99.74 per 1,000 population). These settlements face significant dual challenges: deep socio-economic disadvantage combined with a disproportionately high burden of poverty-related disabilities.

**What constitutes a 'good' outlier settlement?**
A 'good' outlier settlement is defined as one with a low socio-economic index score (below the 25th percentile, which is -0.78) but an unexpectedly low poverty-related disability rate (below the 25th percentile, which is 51.52 per 1,000 population). These settlements demonstrate a degree of resilience, managing to keep poverty-related disability rates low despite their socio-economic challenges.

**How do 'bad' and 'good' outliers compare within socio-economic clusters?**
Both 'bad' and 'good' outliers tend to be found within the lower socio-economic clusters (e.g., cluster 2 or 3), indicating that both types of outliers originate from disadvantaged communities. The key difference lies in their poverty-related disability rates, which are significantly divergent despite similar socio-economic starting points.

### Data Analysis Key Findings

*   **Outlier Definition Thresholds**: The 25th percentile (Q1) for the socio-economic index score was -0.78, and for the poverty-related disability rate was 51.52 per 1,000. The 75th percentile (Q3) for the socio-economic index score was 0.73, and for the poverty-related disability rate was 99.74 per 1,000. These thresholds were used to classify settlements.
*   **'Bad Outlier' Characteristics**: Identified 'Bad Outliers' like 'כפר מנדא' and 'ג'סר א-זרקא' typically exhibited socio-economic index scores around -1.45 and poverty-related disability rates exceeding 100.56 per 1,000. These settlements are often smaller to medium-sized and fall into lower socio-economic clusters.
*   **'Good Outlier' Characteristics**: Identified 'Good Outliers' such as 'קצר א-סר' and 'קריית יערים' displayed socio-economic index scores as low as -2.41 but maintained surprisingly low poverty-related disability rates, around 48.09 per 1,000. These also tend to be in lower socio-economic clusters.
*   **Visualization Confirmation**: An interactive scatter plot visually confirmed that 'Bad Outliers' clustered in the bottom-right of the plot (low socio-economic index, high disability rate), while 'Good Outliers' were situated in the bottom-left (low socio-economic index, low disability rate), with 'Normal' settlements filling the central distribution.

### Insights or Next Steps

*   **Targeted Intervention for 'Bad Outliers'**: 'Bad Outlier' settlements represent critical areas for comprehensive, integrated interventions addressing both socio-economic upliftment and enhanced disability support, as they face compounding disadvantages.
*   **Learn from 'Good Outliers'**: Further qualitative and quantitative research into 'Good Outlier' settlements could uncover protective factors or successful local strategies (e.g., effective community programs, unique demographic resilience) that enable lower poverty-related disability rates despite overall disadvantage, offering replicable models for other communities.
